# Evaluate Voxforge

In [1]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

from qdrant_client import QdrantClient, models

from audio.conversion import conversion
from audio.processing import resample, process_audio
from ai.embedding import embedding_batch
import torch
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device utilisé : {device}")
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))
print(torch.cuda.memory_allocated())


d:\Projet\Voice-Recognition\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
d:\Projet\Voice-Recognition\.venv\lib\site-packages\speechbrain\utils\checkpoints.py:202: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_saf

Device utilisé : cuda
True
NVIDIA GeForce GTX 1080 Ti
89760256


In [2]:
VOXFORGE_PATH = r"D:\Projet\Voice-Recognition\voxforge-fr"
COLLECTION_NAME = "voice_data_base"
BATCH_SIZE = 32  # nombre d'audios envoyés au GPU en une seule fois

client = QdrantClient(host="localhost", port=6333)


## Parse metadata

In [3]:
def parse_readme(readme_path):
    metadata = {"username": None, "gender": None, "age": None, "dialect": None}

    if not readme_path.exists():
        return metadata

    with open(readme_path, "r", encoding="utf-8", errors="ignore") as f:
        content = f.read()

    for line in content.splitlines():
        if line.startswith("User Name:"):
            metadata["username"] = line.replace("User Name:", "").strip()
        elif line.startswith("Gender:"):
            metadata["gender"] = line.replace("Gender:", "").strip()
        elif line.startswith("Age Range:"):
            metadata["age"] = line.replace("Age Range:", "").strip()
        elif line.startswith("Pronunciation dialect:"):
            metadata["dialect"] = line.replace("Pronunciation dialect:", "").strip()
        elif line.startswith("Microphone make:"):
            metadata["micro_make"] = line.replace("Microphone make:", "").strip()
        elif line.startswith("Microphone type:"):
            metadata["micro_type"] = line.replace("Microphone type:", "").strip()
        elif line.startswith("Sampling Rate:"):
            metadata["sampling_rate"] = line.replace("Sampling Rate:", "").strip()

    return metadata


## Collect speakers

In [4]:
speakers = {}

for folder in Path(VOXFORGE_PATH).iterdir():
    if not folder.is_dir():
        continue

    wav_dir = folder / "wav"
    readme = folder / "etc" / "README"

    if not wav_dir.exists():
        continue

    metadata = parse_readme(readme)
    username = metadata["username"]

    if username is None:
        continue

    wavs = sorted(list(wav_dir.glob("*.wav")))
    if len(wavs) < 2:
        continue

    if username not in speakers:
        speakers[username] = {
            "wavs": [],
            "gender": metadata["gender"],
            "age": metadata["age"],
            "dialect": metadata["dialect"],
            "micro_make": metadata["micro_make"],
            "micro_type": metadata["micro_type"],
            "sampling_rate": metadata["sampling_rate"]
        }

    speakers[username]["wavs"].extend(wavs)

print("Speakers:", len(speakers))


Speakers: 332


## Build task list

In [5]:
tasks = []
for username, data in speakers.items():
    wavs = sorted(list(set(data["wavs"])))
    if len(wavs) < 2:
        continue
    for wav_file in wavs[1:]:
        tasks.append((username, data, wav_file))

print("TOTAL TESTS:", len(tasks))


TOTAL TESTS: 23420


## Step 1 — Preprocessing CPU (threads)

I/O + conversion + resample + VAD + denoise sont tous CPU-bound.
On les parallélise en threads pour saturer le CPU pendant que le GPU est libre.

In [10]:
def preprocess(task):
    """Charge et prétraite un fichier audio. Retourne None si pas de voix détectée."""
    username, data, wav_file = task
    try:
        with open(wav_file, "rb") as f:
            audio_bytes = f.read()

        raw = conversion(audio_bytes)
        audio, sr = resample(raw)
        audio, issue = process_audio(audio, sr)

        if issue[0]:
            return None

        return {
            "audio": audio,
            "username": username,
            "gender": data["gender"],
            "age": data["age"],
            "dialect": data["dialect"],
            "micro_make": data["micro_make"],
            "micro_type": data["micro_type"],
            "sampling_rate": data["sampling_rate"]
        }
    except Exception as e:
        print(f"ERROR preprocessing {wav_file}: {e}")
        return None


In [11]:
print("Preprocessing (CPU)...")
preprocessed = []

with ThreadPoolExecutor(max_workers=6) as executor:
    futures = [executor.submit(preprocess, task) for task in tasks]
    for future in tqdm(as_completed(futures), total=len(futures), desc="Preprocessing"):
        r = future.result()
        if r is not None:



            
            preprocessed.append(r)

print(f"Preprocessed: {len(preprocessed)} valid samples")


Preprocessing (CPU)...


Preprocessing: 100%|██████████| 23420/23420 [40:48<00:00,  9.57it/s] 


Preprocessed: 23380 valid samples


## Step 2 — Inférence GPU (batch)

On envoie les audios au GPU par tranches de `BATCH_SIZE`.
Une seule forward pass ECAPA-TDNN pour N audios à la fois.

In [12]:
print("Embedding (GPU)...")
results = []

for i in tqdm(range(0, len(preprocessed), BATCH_SIZE), desc="Embedding batches"):
    batch = preprocessed[i : i + BATCH_SIZE]

    audio_list = [item["audio"] for item in batch]
    embeddings = embedding_batch(audio_list)  # single GPU forward pass

    for item, emb in zip(batch, embeddings):
        try:
            result = client.query_points(
                collection_name=COLLECTION_NAME,
                query=emb,
                limit=1,
                search_params=models.SearchParams(hnsw_ef=200, exact=False),
            )

            if len(result.points) == 0:
                continue

            point = result.points[0]
            predicted = point.payload["username"]

            results.append({
                "speaker": item["username"],
                "predicted": predicted,
                "score": float(point.score),
                "correct": predicted == item["username"],
                "gender": item["gender"],
                "age": item["age"],
                "dialect": item["dialect"],
                "micro_make": item["micro_make"],
                "micro_type": item["micro_type"],
                "sampling_rate": item["sampling_rate"]
            })

        except Exception as e:
            print(f"ERROR qdrant query: {e}")

results_df = pd.DataFrame(results)
results_df.to_csv("voxforge_results.csv", index=False)
print("DONE")


Embedding (GPU)...


Embedding batches:   0%|          | 0/731 [00:00<?, ?it/s]

Embedding batches: 100%|██████████| 731/731 [07:21<00:00,  1.66it/s]


DONE
